# 07-units — Dimensionally-constrained equation discovery (Bogotá, PM2.5)

Per the standing TODO (deferred earlier as "would be a deliberate choice to add later,
not a quick fix"): the equations discovered so far are NOT dimensionally consistent --
PySR treats every input as a bare number, so e.g. the O3 equation's
`sqrt(photochemical_potential/no)` term reduces to `sqrt(W/m2)` after concentration
units cancel, which isn't a valid concentration unit on its own. PySR supports a
unit-aware search mode (`X_units`/`y_units` in `.fit()`, backed by DynamicQuantities.jl)
that restricts `+`/`-` to same-unit operands and `sqrt`/`log`/`exp` to dimensionless (or,
for sqrt, perfect-square-unit) arguments -- enforcing real dimensional homogeneity
structurally, not just checking it after the fact.

**Units assigned** (documented assumption, since some are genuinely ambiguous -- see
MASTER.md's units discussion):
- `pm25`, `co`: mass concentration, `kg/m^3` (µg/m3 in practice, but PySR's unit checker
  only cares about dimension, not the numeric scale factor) -- this assumes CO is a mass
  concentration like PM2.5, matching this project's documented `qc.py` convention. If the
  true convention for gas species is actually a dimensionless mixing ratio (ppb) instead,
  this assumption would need revisiting -- flagged as a known ambiguity, not resolved
  here.
- `w700`: ERA5 vertical velocity (omega) at 700 hPa, `Pa/s` (pressure-coordinate
  convention, not m/s).
- `wind_speed_era5`: `m/s`.
- `tcwv_lag1`: total column water vapour, `kg/m^2`.
- `precip_total_mm_lag1`: `mm` (a scaled length).
- `radiation_mean`: `W/m^2`.
- `hotspots_sector_E_lag3`, `hotspots_sector_N_lag2`: dimensionless counts (`""`).

Uses the SAME no-lag feature selection as the unconstrained `07_equation_discovery.ipynb`
for a direct, apples-to-apples comparison -- same features, same target, only the
dimensional-consistency constraint differs.


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
FIG_DIR = PROJECT_ROOT / "Scripts" / "colombia" / "bogota" / "figures"

df = pd.read_parquet(DATA_DIR / "bogota_master_dataset.parquet")
df = df.dropna(subset=['pm25']).reset_index(drop=True)

with open(DATA_DIR / "bogota_feature_selection.json") as f:
    selection = json.load(f)
features = selection['selected_no_lag']
print("Using features:", features)

data = df.dropna(subset=features + ['pm25']).reset_index(drop=True)
X = data[features].to_numpy()
y = data['pm25'].to_numpy()
print(data.shape)


Using features: ['w700', 'co', 'hotspots_sector_E_lag3', 'tcwv_lag1', 'precip_total_mm_lag1', 'wind_speed_era5', 'radiation_mean', 'hotspots_sector_N_lag2']
(3282, 215)


## Assign units (documented assumptions above)

In [2]:
UNIT_MAP = {
    'w700': 'Pa/s',
    'co': 'kg/m^3',
    'hotspots_sector_E_lag3': '',
    'tcwv_lag1': 'kg/m^2',
    'precip_total_mm_lag1': 'mm',
    'wind_speed_era5': 'm/s',
    'radiation_mean': 'W/m^2',
    'hotspots_sector_N_lag2': '',
}
X_units = [UNIT_MAP[f] for f in features]
y_units = 'kg/m^3'
print("X_units:", dict(zip(features, X_units)))
print("y_units:", y_units)


X_units: {'w700': 'Pa/s', 'co': 'kg/m^3', 'hotspots_sector_E_lag3': '', 'tcwv_lag1': 'kg/m^2', 'precip_total_mm_lag1': 'mm', 'wind_speed_era5': 'm/s', 'radiation_mean': 'W/m^2', 'hotspots_sector_N_lag2': ''}
y_units: kg/m^3


## Unit-constrained symbolic regression

In [3]:
from pysr import PySRRegressor

sr_model_units = PySRRegressor(
    niterations=200,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["log", "sqrt", "exp"],
    model_selection="best",
    maxsize=25,
    populations=24,
)
sr_model_units.fit(X, y, variable_names=features, X_units=X_units, y_units=y_units)
print(sr_model_units)


Compiling Julia backend...


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.210e+04
Progress: 56 / 200 total iterations (28.271%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.444e+01  0.000e+00  y[m⁻³ kg] = 18.294[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.879[?]
5           3.982e+01  8.401e-04  y[m⁻³ kg] = (co[m⁻³ kg] * 22.673[?]) + 1.0175[?]
6           3.364e+01  1.684e-01  y[m⁻³ kg] = sqrt(co[m⁻³ kg] / wind_speed_era5[m s⁻...
                                              ¹]) * 20.581[?]
7           3.169e+01  5.984e-02  y[m⁻³ kg] = co[m⁻³ kg] * (23.187[?] - (w700[s⁻¹ Pa...
                                              ] * 25.471[?]))
8           3.092e+01  2.456e-02  y[m⁻³ kg] = sqrt(co[m⁻³ kg]) * (20.987[?] - (w700[...
                                              s⁻¹ Pa] * 31.892[?]))
9

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          18.293705   
	1         0.239907                                      co * 23.86051   
	2         0.000860                       (co * 22.673256) + 0.9745013   
	3   >>>>  0.168407             sqrt(co / wind_speed_era5) * 20.612934   
	4         0.061423              co * ((w700 * -27.91419) + 23.261927)   
	5         0.023492        sqrt(co) * (21.068121 - (w700 * 31.923283))   
	6         0.034340  (co * (16.971088 - (w700 * 31.880013))) + 5.10...   
	7         0.023563  (co + (((0.20026642 - w700) * 25.387484) / tcw...   
	8         0.028617  (co + ((0.29734862 - w700) * (16.451641 / (tcw...   
	9         0.000973  ((((0.29734862 - w700) * 16.203215) / (wind_sp...   
	10        0.045610  (co + (((0.29734862 - w700) / (wind_speed_era5...   
	11        0.000034  (co + (((0.29734862 - w700) / (wind_speed_era5...   
	12      

In [4]:
sr_pareto_units = sr_model_units.equations_[['complexity', 'loss', 'equation']]
sr_pareto_units['r2'] = sr_pareto_units.apply(
    lambda row: r2_score(y, sr_model_units.predict(X, index=row.name)), axis=1
)
sr_pareto_units


,complexity,loss,equation,r2
0,1,64.440890,18.293705,-9.191102e-08
1,3,39.882347,co * 23.86051,3.811018e-01
2,5,39.813820,(co * 22.673256) + 0.9745013,3.821653e-01
3,6,33.643078,sqrt(co / wind_speed_era5) * 20.612934,4.779234e-01
4,7,31.638792,co * ((w700 * -27.91419) + 23.261927),5.090261e-01
5,8,30.904210,sqrt(co) * (21.068121 - (w700 * 31.923283)),5.204254e-01
6,9,29.860977,(co * (16.971088 - (w700 * 31.880013))) + 5.10...,5.366145e-01
7,11,28.486403,(co + (((0.20026642 - w700) * 25.387484) / tcw...,5.579452e-01
8,13,26.901781,(co + ((0.29734862 - w700) * (16.451641 / (tcw...,5.825355e-01
9,17,26.797276,((((0.29734862 - w700) * 16.203215) / (wind_sp...,5.841572e-01


## Compare against the unconstrained equation

Same features/target, loaded from the already-saved unconstrained Pareto front, for a
direct R2-vs-dimensional-consistency tradeoff comparison.


In [5]:
unconstrained = pd.read_csv(DATA_DIR / "bogota_pysr_pareto_front.csv")
best_unconstrained_r2 = None
if 'r2' in unconstrained.columns:
    best_unconstrained_r2 = unconstrained['r2'].max()
else:
    # loss-based fallback: R2 = 1 - loss/var(y) for a plain-MSE loss
    var_y = np.var(y)
    best_unconstrained_r2 = 1 - unconstrained['loss'].min() / var_y

best_units_r2 = sr_pareto_units['r2'].max()
print(f"Best unconstrained PySR R2 (no units): {best_unconstrained_r2:.3f}")
print(f"Best unit-constrained PySR R2:         {best_units_r2:.3f}")
print(f"Cost of dimensional consistency: {best_unconstrained_r2 - best_units_r2:+.3f} R2")


Best unconstrained PySR R2 (no units): 0.601
Best unit-constrained PySR R2:         0.604
Cost of dimensional consistency: -0.003 R2


## Save results

In [6]:
sr_model_units.equations_.to_csv(DATA_DIR / "bogota_pm25_pysr_units_pareto_front.csv", index=False)
print("Saved unit-constrained PySR Pareto front")


Saved unit-constrained PySR Pareto front
